In [ ]:
using Bessels, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl")
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi
ω_b_0 = 2*ω_a 

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=ω_b_0
ω_p=2ω_a-ω_d

ϵ_0 = 0.9#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=19.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
#ATS + Mean_Field + ladder_operator
function equation_stark!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    sin_pump =  - 2*E_J*sin(ϵ_0)
    coupling = (cos((a+conj(a))/sqrt(2) * sqrt(Z_a)/ϕ_quantum + (b+conj(b))/sqrt(2) * sqrt(Z_b)/ϕ_quantum) - 0) / ϕ_quantum / sqrt(2)
        
    du[1] = -1im*(ω_a*a + sqrt(Z_a)*sin_pump*coupling) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) + sqrt(Z_b)*sin_pump*coupling) - p.environment_prepared.κ_b * b
    return nothing
end

In [ ]:
t_max=200/κ_b
a_0 = 0.0
b_0 = 0.0

ϵ_0 = 0.3
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d_over_g2 = 9
ϵ_d = ϵ_d_over_g2*g2

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
#a, b, t_list = simulator(t_max, system, drive, environment, equation = equation_stark!);
system_prepared = prepare(system)
drive_prepared = prepare(drive)
environment_prepared = prepare(system,environment)

dt = 1/20
u_0_tot = get_init(system_prepared, environment_prepared)

p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

a, b, t_list = default_solver(u_0_tot, (0.0, t_max), p, dt; equation=equation_stark!)

In [ ]:
F, freqs = FFT_perso(real.(b), t_list)
mask = (freqs .> 7) .& (freqs .< 8.5)
F = F[mask]
freqs = freqs[mask]
freq_max = abs(freqs[argmax(abs.(F))])
threshold = 0.05 #GHz
F[argmax(abs.(F))] = 0
freq_max_bis = abs(freqs[argmax(abs.(F))])
while abs(freq_max_bis-freq_max)<threshold
    F[argmax(abs.(F))] = 0
    freq_max_bis = abs(freqs[argmax(abs.(F))])
end

In [ ]:
freq_max_bis

In [ ]:
F, freqs = FFT_perso(real.(b), t_list)
plot(freqs, abs.(F), linewidth=.1)
vline!([freq_max])
vline!([freq_max_bis])
plot!(xlims=(7,9.5), yaxis=:log)

In [ ]:
t_max=200/κ_b
a_0 = 0.0
b_0 = 0.0
ω_d = freq_max_bis*2pi
ω_p = 0.0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a, b, t_list = simulator(t_max, system, drive, environment, equation = equation_stark!);

In [ ]:
F, freqs = FFT_perso(real.(b), t_list)
plot(freqs, abs.(F), linewidth=.1)
vline!([freq_max_bis])
plot!(xlims=(7,8.5), yaxis=:log)

In [ ]:
F, freqs = FFT_perso(real.(a), t_list)
#vline([freq_max_bis/2])
plot(freqs, abs.(F), linewidth=.1)
plot!(xlims=(3,4.5), yaxis=:log)

In [ ]:
mask = (freqs .> 3) .& (freqs .< 4.5)
F = F[mask]
freqs = freqs[mask]
freq_max = abs(freqs[argmax(abs.(F))])

In [ ]:
ω_rot = freq_max*2*pi
a_rot = (a.-sum(a)/length(a)).*exp.(1im*ω_rot*t_list);

In [ ]:
plot(real.(a_rot[1:div(length(a_rot),4)].+real(sum(a)/length(a))), imag.(a_rot[1:div(length(a_rot),4)]).+imag(sum(a)/length(a)))

In [ ]:
function find_drive_freq(F, freqs)
    mask = (freqs .> 7) .& (freqs .< 8.5)
    F = F[mask]
    freqs = freqs[mask]
    freq_max = abs(freqs[argmax(abs.(F))])
    threshold = 0.05 #GHz
    F[argmax(abs.(F))] = 0
    freq_max_bis = abs(freqs[argmax(abs.(F))])
    while abs(freq_max_bis-freq_max)<threshold
        F[argmax(abs.(F))] = 0
        freq_max_bis = abs(freqs[argmax(abs.(F))])
    end
    return(freq_max_bis)
end

In [ ]:
Drive_freq = Dict{Float64,Dict{Float64,Tuple{Float64,Float64}}}()
epsilon_d_over_g2_list = range(start = 9, stop = 11, length = 3)
epsilon_p_list = range(start = 0.3, stop = .9, length = 7)
t_max=400/κ_b
a_0 = 0.375
b_0 = 0.0
ω_d = ω_b_0
environment = MeanField(κ_a, κ_b)
system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
system_prepared = prepare(system)
environment_prepared = prepare(system,environment)
u_0_tot = get_init(system_prepared, environment_prepared)
for epsilon_p in ProgressBar(epsilon_p_list)
    g2 = besselj(1,epsilon_p)*E_J*φ_a^2*φ_b # (Leon's paper)
    Drive_freq[epsilon_p] = Dict{Float64,Float64}()
    for epsilon_d_over_g2 in epsilon_d_over_g2_list
        ϵ_d = epsilon_d_over_g2 * g2
        drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, epsilon_p, ω_p, pump_phase)
        drive_prepared = prepare(drive)
        dt = 1/20
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
        _, b, t_list = default_solver(u_0_tot, (0.0, t_max), p, dt; equation=equation_stark!)
        F, freqs = FFT_perso(real.(b), t_list)
        f_d = find_drive_freq(F, freqs)

        drive = DriveTwoModesJA(ϵ_d, f_d*2pi, buffer_phase, ϵ_0, ω_p, pump_phase)
        p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
        a, _, t_list = default_solver(u_0_tot, (0.0, t_max), p, dt; equation=equation_stark!)
        F, freqs = FFT_perso(real.(a), t_list)
        mask = (freqs .> 3) .& (freqs .< 4.5)
        F = F[mask]
        freqs = freqs[mask]
        f_rot = abs(freqs[argmax(abs.(F))])
        Drive_freq[epsilon_p][epsilon_d_over_g2] = (f_d, f_rot)
    end
end

In [ ]:
Drive_freq

In [ ]:
ϵ_p = rand(epsilon_p_list)
g2 = besselj(1,ϵ_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d_over_g2 = rand(epsilon_d_over_g2_list)
ϵ_d = ϵ_d_over_g2*g2

ω_d, ω_rot = 2pi.*Drive_freq[ϵ_p][ϵ_d_over_g2]

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
t_max = 20/κ_b
a, b, t_list = simulator(t_max, system, drive, environment, equation = equation_stark!);
a_rot = (a.-sum(a)/length(a)).*exp.(1im*ω_rot*t_list);

In [ ]:
plot(real.(a_rot[1:div(length(a_rot),1)].+real(sum(a)/length(a))), imag.(a_rot[1:div(length(a_rot),1)]).+imag(sum(a)/length(a)))